In [1]:
!pip install -q sentence-transformers transformers scikit-learn scipy joblib pandas numpy torch

In [2]:
# ============================================================
# Compare MiniLM, SciBERT, SPECTER2, BGE, and TF-IDF
# for predicting annotated Technical Depth
#
# Inputs:
#   1. Annotated sample:
#      /mnt/data/dedup_ranked_outputs_sample_300_seed_4000.csv
#
#   2. Full 2,619-paper corpus:
#      /mnt/data/dedup_ranked_outputs.xlsx - Copy of Sheet1.csv
#
# Required annotated columns:
#   - Paper ID
#   - Canonical Abstract
#   - Average Grade
#
# Output:
#   - model_comparison_metrics.csv
#   - all_2619_predicted_technical_depth_best_model.csv
#   - best_technical_depth_model.joblib
# ============================================================

# Install once if needed:
# !pip install -q sentence-transformers transformers scikit-learn scipy joblib pandas numpy torch

from pathlib import Path
import gc
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr, spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 4000

ANNOTATED_FILE = Path(
    "dedup_ranked_outputs_sample_300_seed_4000.csv"
)

CORPUS_FILE = Path(
    "dedup_ranked_outputs.xlsx - Copy of Sheet1.csv"
)

ID_COL = "Paper ID"
TEXT_COL = "Canonical Abstract"
TARGET_COL = "Average Grade"

N_SPLITS = 5
BATCH_SIZE = 32

OUTPUT_METRICS = Path("model_comparison_metrics.csv")
OUTPUT_PREDICTIONS = Path(
    "all_2619_predicted_technical_depth_best_model.csv"
)
OUTPUT_MODEL = Path("best_technical_depth_model.joblib")
OUTPUT_REPORT = Path("best_technical_depth_model_report.json")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)


# ============================================================
# 2. MODEL DEFINITIONS
# ============================================================

EMBEDDING_MODELS = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",

    # Scientific BERT encoder
    "SciBERT": "allenai/scibert_scivocab_uncased",

    # Scientific document embedding model
    "SPECTER2": "allenai/specter2_base",

    # General-purpose BGE embedding model
    "BGE": "BAAI/bge-small-en-v1.5",
}

RIDGE_ALPHAS = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]


# ============================================================
# 3. LOAD AND CLEAN DATA
# ============================================================

annotated = pd.read_csv(ANNOTATED_FILE, low_memory=False)
corpus = pd.read_csv(CORPUS_FILE, low_memory=False)

required_annotated = {ID_COL, TEXT_COL, TARGET_COL}
required_corpus = {ID_COL, TEXT_COL}

missing_annotated = required_annotated - set(annotated.columns)
missing_corpus = required_corpus - set(corpus.columns)

if missing_annotated:
    raise ValueError(
        f"Annotated file is missing columns: {missing_annotated}"
    )

if missing_corpus:
    raise ValueError(
        f"Corpus file is missing columns: {missing_corpus}"
    )

annotated = annotated.copy()
corpus = corpus.copy()

annotated[TEXT_COL] = (
    annotated[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

corpus[TEXT_COL] = (
    corpus[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.strip()
)

annotated[TARGET_COL] = pd.to_numeric(
    annotated[TARGET_COL],
    errors="coerce",
)

annotated = annotated[
    annotated[ID_COL].notna()
    & annotated[TEXT_COL].ne("")
    & annotated[TARGET_COL].notna()
].drop_duplicates(subset=ID_COL)

corpus = corpus[
    corpus[ID_COL].notna()
].drop_duplicates(subset=ID_COL)

missing_ids = set(annotated[ID_COL]) - set(corpus[ID_COL])

if missing_ids:
    raise ValueError(
        f"{len(missing_ids)} annotated Paper IDs are absent from the full corpus."
    )

print("Annotated rows:", len(annotated))
print("Corpus rows:", len(corpus))
print("Target range:", annotated[TARGET_COL].min(), annotated[TARGET_COL].max())


# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "RMSE": float(
            mean_squared_error(y_true, y_pred) ** 0.5
        ),
        "MAE": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "R2": float(
            r2_score(y_true, y_pred)
        ),
        "Pearson": float(
            pearsonr(y_true, y_pred).statistic
        ),
        "Spearman": float(
            spearmanr(y_true, y_pred).statistic
        ),
    }


# ============================================================
# 5. CROSS-VALIDATED RIDGE FOR DENSE EMBEDDINGS
# ============================================================

def evaluate_dense_embeddings(
    model_name,
    embeddings,
    targets,
    cv,
):
    results = []
    predictions_by_alpha = {}

    for alpha in RIDGE_ALPHAS:
        oof_predictions = np.zeros(len(targets), dtype=float)

        for train_idx, valid_idx in cv.split(embeddings):
            X_train = embeddings[train_idx]
            X_valid = embeddings[valid_idx]

            y_train = targets[train_idx]

            model = Pipeline([
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(alpha=alpha),
                ),
            ])

            model.fit(X_train, y_train)
            oof_predictions[valid_idx] = model.predict(X_valid)

        metrics = calculate_metrics(
            targets,
            oof_predictions,
        )

        results.append({
            "Model": model_name,
            "Alpha": alpha,
            **metrics,
        })

        predictions_by_alpha[alpha] = oof_predictions

    result_df = pd.DataFrame(results)

    best_row = result_df.sort_values(
        ["RMSE", "MAE"],
        ascending=[True, True],
    ).iloc[0]

    best_alpha = float(best_row["Alpha"])
    best_predictions = predictions_by_alpha[best_alpha]

    return result_df, best_alpha, best_predictions


# ============================================================
# 6. CROSS-VALIDATED TF-IDF
# ============================================================

def evaluate_tfidf(
    texts,
    targets,
    cv,
):
    results = []
    predictions_by_alpha = {}

    for alpha in RIDGE_ALPHAS:
        oof_predictions = np.zeros(len(targets), dtype=float)

        for train_idx, valid_idx in cv.split(texts):
            X_train = texts.iloc[train_idx]
            X_valid = texts.iloc[valid_idx]

            y_train = targets[train_idx]

            pipeline = Pipeline([
                (
                    "tfidf",
                    TfidfVectorizer(
                        lowercase=True,
                        strip_accents="unicode",
                        ngram_range=(1, 2),
                        min_df=2,
                        max_df=0.98,
                        max_features=30000,
                        sublinear_tf=True,
                    ),
                ),
                (
                    "ridge",
                    Ridge(alpha=alpha),
                ),
            ])

            pipeline.fit(X_train, y_train)
            oof_predictions[valid_idx] = pipeline.predict(X_valid)

        metrics = calculate_metrics(
            targets,
            oof_predictions,
        )

        results.append({
            "Model": "TF-IDF",
            "Alpha": alpha,
            **metrics,
        })

        predictions_by_alpha[alpha] = oof_predictions

    result_df = pd.DataFrame(results)

    best_row = result_df.sort_values(
        ["RMSE", "MAE"],
        ascending=[True, True],
    ).iloc[0]

    best_alpha = float(best_row["Alpha"])
    best_predictions = predictions_by_alpha[best_alpha]

    return result_df, best_alpha, best_predictions


# ============================================================
# 7. GENERATE EMBEDDINGS
# ============================================================

annotated_texts = annotated[TEXT_COL].tolist()
targets = annotated[TARGET_COL].to_numpy(dtype=float)

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

all_metric_rows = []
best_configs = {}
annotated_embeddings = {}

for short_name, huggingface_name in EMBEDDING_MODELS.items():

    print("\n" + "=" * 70)
    print("Embedding model:", short_name)
    print("Hugging Face model:", huggingface_name)
    print("=" * 70)

    model = SentenceTransformer(
        huggingface_name,
        device=DEVICE,
    )

    embeddings = model.encode(
        annotated_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    print("Embedding shape:", embeddings.shape)

    annotated_embeddings[short_name] = embeddings

    result_df, best_alpha, best_oof = evaluate_dense_embeddings(
        model_name=short_name,
        embeddings=embeddings,
        targets=targets,
        cv=cv,
    )

    all_metric_rows.append(result_df)

    best_configs[short_name] = {
        "alpha": best_alpha,
        "oof_predictions": best_oof,
        "hf_model": huggingface_name,
        "embedding_dimension": int(embeddings.shape[1]),
    }

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 8. EVALUATE TF-IDF
# ============================================================

print("\n" + "=" * 70)
print("Evaluating TF-IDF")
print("=" * 70)

tfidf_results, tfidf_best_alpha, tfidf_best_oof = evaluate_tfidf(
    texts=annotated[TEXT_COL],
    targets=targets,
    cv=cv,
)

all_metric_rows.append(tfidf_results)

best_configs["TF-IDF"] = {
    "alpha": tfidf_best_alpha,
    "oof_predictions": tfidf_best_oof,
}


# ============================================================
# 9. SELECT BEST MODEL
# ============================================================

all_metrics = pd.concat(
    all_metric_rows,
    ignore_index=True,
)

all_metrics = all_metrics.sort_values(
    ["RMSE", "MAE", "Spearman"],
    ascending=[True, True, False],
).reset_index(drop=True)

all_metrics.to_csv(
    OUTPUT_METRICS,
    index=False,
)

print("\nComplete model comparison:")
print(all_metrics.to_string(index=False))

best_row = all_metrics.iloc[0]

best_model_name = best_row["Model"]
best_alpha = float(best_row["Alpha"])

print("\nBest model:", best_model_name)
print("Best alpha:", best_alpha)
print("Best RMSE:", best_row["RMSE"])
print("Best MAE:", best_row["MAE"])
print("Best R2:", best_row["R2"])
print("Best Pearson:", best_row["Pearson"])
print("Best Spearman:", best_row["Spearman"])


# ============================================================
# 10. TRAIN BEST MODEL ON ALL 300 LABELLED PAPERS
# ============================================================

corpus_texts = corpus[TEXT_COL].tolist()

if best_model_name == "TF-IDF":

    final_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                max_features=30000,
                sublinear_tf=True,
            ),
        ),
        (
            "ridge",
            Ridge(alpha=best_alpha),
        ),
    ])

    final_model.fit(
        annotated[TEXT_COL],
        targets,
    )

    corpus_predictions = final_model.predict(
        corpus[TEXT_COL]
    )

    saved_object = {
        "model_type": "TF-IDF",
        "pipeline": final_model,
        "target_column": TARGET_COL,
        "text_column": TEXT_COL,
        "id_column": ID_COL,
    }

else:

    hf_model_name = best_configs[best_model_name]["hf_model"]

    embedding_model = SentenceTransformer(
        hf_model_name,
        device=DEVICE,
    )

    train_embeddings = annotated_embeddings[best_model_name]

    corpus_embeddings = embedding_model.encode(
        corpus_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    final_regressor = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=best_alpha),
        ),
    ])

    final_regressor.fit(
        train_embeddings,
        targets,
    )

    corpus_predictions = final_regressor.predict(
        corpus_embeddings
    )

    saved_object = {
        "model_type": best_model_name,
        "huggingface_model": hf_model_name,
        "regressor": final_regressor,
        "target_column": TARGET_COL,
        "text_column": TEXT_COL,
        "id_column": ID_COL,
        "normalize_embeddings": True,
    }


# ============================================================
# 11. CLIP AND NORMALIZE PREDICTIONS
# ============================================================

human_min = float(targets.min())
human_max = float(targets.max())

corpus_predictions = np.clip(
    corpus_predictions,
    human_min,
    human_max,
)

prediction_min = float(corpus_predictions.min())
prediction_max = float(corpus_predictions.max())

if prediction_max > prediction_min:
    technical_depth_norm = (
        corpus_predictions - prediction_min
    ) / (
        prediction_max - prediction_min
    )
else:
    technical_depth_norm = np.zeros(
        len(corpus_predictions)
    )


# ============================================================
# 12. ADD ANNOTATED AND OOF VALUES
# ============================================================

best_oof_predictions = best_configs[
    best_model_name
]["oof_predictions"]

annotation_map = annotated.set_index(
    ID_COL
)[TARGET_COL]

oof_map = pd.Series(
    best_oof_predictions,
    index=annotated[ID_COL],
)

output = corpus.copy()

output["Predicted Technical Depth"] = corpus_predictions
output["Technical Depth Norm"] = technical_depth_norm

output["Annotated Technical Depth"] = output[
    ID_COL
].map(annotation_map)

output["OOF Predicted Technical Depth"] = output[
    ID_COL
].map(oof_map)

output["Technical Depth Source"] = np.where(
    output["Annotated Technical Depth"].notna(),
    "annotated-training-sample",
    "model-predicted",
)

output["Technical Depth Model"] = best_model_name

output.to_csv(
    OUTPUT_PREDICTIONS,
    index=False,
)

joblib.dump(
    saved_object,
    OUTPUT_MODEL,
)


# ============================================================
# 13. SAVE REPORT
# ============================================================

report = {
    "training_rows": int(len(annotated)),
    "corpus_rows": int(len(corpus)),
    "target_column": TARGET_COL,
    "best_model": best_model_name,
    "best_alpha": best_alpha,
    "selection_rule": (
        "Lowest RMSE, then lowest MAE, then highest Spearman correlation"
    ),
    "best_metrics": {
        "RMSE": float(best_row["RMSE"]),
        "MAE": float(best_row["MAE"]),
        "R2": float(best_row["R2"]),
        "Pearson": float(best_row["Pearson"]),
        "Spearman": float(best_row["Spearman"]),
    },
    "human_score_range": [
        human_min,
        human_max,
    ],
    "predicted_score_range": [
        prediction_min,
        prediction_max,
    ],
    "outputs": {
        "metrics": str(OUTPUT_METRICS),
        "predictions": str(OUTPUT_PREDICTIONS),
        "model": str(OUTPUT_MODEL),
    },
}

OUTPUT_REPORT.write_text(
    json.dumps(
        report,
        indent=2,
    )
)

print("\nSaved:")
print(OUTPUT_METRICS)
print(OUTPUT_PREDICTIONS)
print(OUTPUT_MODEL)
print(OUTPUT_REPORT)

Device: cpu
Annotated rows: 300
Corpus rows: 2619
Target range: 2.0 8.5

Embedding model: MiniLM
Hugging Face model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (300, 384)

Evaluating TF-IDF

Complete model comparison:
   Model  Alpha     RMSE      MAE       R2  Pearson  Spearman
  MiniLM  30.00 0.278450 0.224343 0.919553 0.959065  0.943409
  MiniLM  10.00 0.314414 0.253634 0.897431 0.949303  0.927636
  MiniLM   3.00 0.360223 0.289393 0.865366 0.936569  0.904488
  MiniLM   1.00 0.391388 0.314401 0.841061 0.927293  0.889422
  MiniLM   0.30 0.408760 0.327302 0.826639 0.921873  0.880168
  MiniLM   0.10 0.414892 0.331889 0.821399 0.919917  0.877688
  MiniLM   0.01 0.417874 0.334107 0.818822 0.918953  0.876313
     BGE  30.00 0.514934 0.407817 0.724883 0.853984  0.802792
SPECTER2  30.00 0.567098 0.453641 0.666320 0.824950  0.748020
  TF-IDF   0.01 0.596965 0.479258 0.630246 0.811244  0.725407
     BGE  10.00 0.598312 0.479582 0.628576 0.809336  0.751265
  TF-IDF   0.10 0.603442 0.485023 0.622180 0.812175  0.727916
 SciBERT  30.00 0.611940 0.474148 0.611464 0.792297  0.708713
  TF-IDF   0.30 0.620284 0.498606 0.600796 0.811206  0.72

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/82 [00:00<?, ?it/s]


Saved:
model_comparison_metrics.csv
all_2619_predicted_technical_depth_best_model.csv
best_technical_depth_model.joblib
best_technical_depth_model_report.json


In [8]:
# ============================================================
# CRITIC weighting for the final 4-input paper score
#
# Inputs:
#   1. Taxonomy-based relevance
#   2. Predicted technical depth
#   3. Citation impact
#   4. Query coverage
#
# The CRITIC weights:
#   - are non-negative
#   - sum to 1
#
# Since all four criteria are normalized to [0,1],
# the weighted Final Score is also bounded within [0,1].
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. FILE SETTINGS
# ============================================================

INPUT_FILE = Path(
    "dedup_ranked_outputs.xlsx - Copy of Sheet1 (2).csv"
)

OUTPUT_FILE = Path(
    "all_2619_CRITIC_final_scores.csv"
)

WEIGHTS_FILE = Path(
    "CRITIC_weights.csv"
)

CORRELATION_FILE = Path(
    "CRITIC_correlation_matrix.csv"
)

REPORT_FILE = Path(
    "CRITIC_report.json"
)


# ============================================================
# 2. COLUMN SETTINGS
# ============================================================

ID_COL = "Paper ID"

# Four and only four CRITIC inputs
RELEVANCE_COL = "max_taxonomy_score"
TECHNICAL_DEPTH_COL = "Technical Depth Norm"
CITATION_COL = "Max Citations"
QUERY_COUNT_COL = "Number of Queries Matched"


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False,
)

required_columns = {
    ID_COL,
    RELEVANCE_COL,
    TECHNICAL_DEPTH_COL,
    CITATION_COL,
    QUERY_COUNT_COL,
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

df = df.copy()


# ============================================================
# 4. CONVERT INPUTS TO NUMERIC
# ============================================================

numeric_columns = [
    RELEVANCE_COL,
    TECHNICAL_DEPTH_COL,
    CITATION_COL,
    QUERY_COUNT_COL,
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )


# ============================================================
# 5. CHECK PAPER IDS
# ============================================================

if df[ID_COL].isna().any():
    missing_id_count = int(
        df[ID_COL].isna().sum()
    )

    raise ValueError(
        f"{missing_id_count} rows have missing Paper IDs."
    )

if df[ID_COL].duplicated().any():
    duplicate_count = int(
        df[ID_COL].duplicated().sum()
    )

    raise ValueError(
        f"{duplicate_count} duplicated Paper IDs were found."
    )


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

# Missing citation counts are treated as zero citations.
df[CITATION_COL] = (
    df[CITATION_COL]
    .fillna(0)
    .clip(lower=0)
)

# Missing query counts are treated as zero matches.
df[QUERY_COUNT_COL] = (
    df[QUERY_COUNT_COL]
    .fillna(0)
    .clip(lower=0)
)

# Relevance and technical depth are substantive model criteria.
# Missing values are imputed using their medians.
if df[RELEVANCE_COL].notna().sum() == 0:
    raise ValueError(
        f"Column '{RELEVANCE_COL}' contains no valid numeric values."
    )

if df[TECHNICAL_DEPTH_COL].notna().sum() == 0:
    raise ValueError(
        f"Column '{TECHNICAL_DEPTH_COL}' contains no valid numeric values."
    )

df[RELEVANCE_COL] = df[
    RELEVANCE_COL
].fillna(
    df[RELEVANCE_COL].median()
)

df[TECHNICAL_DEPTH_COL] = df[
    TECHNICAL_DEPTH_COL
].fillna(
    df[TECHNICAL_DEPTH_COL].median()
)


# ============================================================
# 7. NORMALIZATION FUNCTION
# ============================================================

def min_max_normalize(series: pd.Series) -> pd.Series:
    """
    Normalize a numeric Series to [0,1].

    Formula:
        (x - min(x)) / (max(x) - min(x))

    If all values are identical, the criterion has no
    discriminatory power, so the function returns zeros.
    """

    minimum = float(series.min())
    maximum = float(series.max())

    if not np.isfinite(minimum):
        raise ValueError(
            f"Invalid minimum in column '{series.name}'."
        )

    if not np.isfinite(maximum):
        raise ValueError(
            f"Invalid maximum in column '{series.name}'."
        )

    if np.isclose(
        maximum,
        minimum,
    ):
        return pd.Series(
            np.zeros(
                len(series),
                dtype=float,
            ),
            index=series.index,
            name=series.name,
        )

    normalized = (
        series - minimum
    ) / (
        maximum - minimum
    )

    return normalized.clip(
        lower=0.0,
        upper=1.0,
    )


# ============================================================
# 8. CREATE THE FOUR NORMALIZED CRITERIA
# ============================================================

# ------------------------------------------------------------
# Criterion 1: Taxonomy-based relevance
# ------------------------------------------------------------

df["Relevance Norm"] = min_max_normalize(
    df[RELEVANCE_COL]
)


# ------------------------------------------------------------
# Criterion 2: Technical depth
# ------------------------------------------------------------

# Although this column is expected to already be normalized,
# it is normalized again defensively to guarantee [0,1].
df["Technical Depth Norm CRITIC"] = min_max_normalize(
    df[TECHNICAL_DEPTH_COL]
)


# ------------------------------------------------------------
# Criterion 3: Citation impact
# ------------------------------------------------------------

# Log transformation reduces domination by highly cited outliers.
df["Citation Log"] = np.log1p(
    df[CITATION_COL]
)

df["Citation Norm"] = min_max_normalize(
    df["Citation Log"]
)


# ------------------------------------------------------------
# Criterion 4: Query coverage
# ------------------------------------------------------------

df["Query Coverage Norm"] = min_max_normalize(
    df[QUERY_COUNT_COL]
)


# ============================================================
# 9. BUILD THE CRITIC DECISION MATRIX
# ============================================================

criteria_columns = [
    "Relevance Norm",
    "Technical Depth Norm CRITIC",
    "Citation Norm",
    "Query Coverage Norm",
]

X = df[
    criteria_columns
].astype(float).copy()


# ============================================================
# 10. VALIDATE NORMALIZED CRITERIA
# ============================================================

if X.isna().any().any():
    bad_columns = X.columns[
        X.isna().any()
    ].tolist()

    raise ValueError(
        "Missing values remain in normalized criteria: "
        f"{bad_columns}"
    )

if not np.isfinite(
    X.to_numpy()
).all():
    raise ValueError(
        "The CRITIC decision matrix contains infinite "
        "or invalid numeric values."
    )

outside_range = (
    (X < -1e-12)
    | (X > 1 + 1e-12)
)

if outside_range.any().any():
    bad_columns = X.columns[
        outside_range.any()
    ].tolist()

    raise ValueError(
        "The following normalized criteria contain values "
        f"outside [0,1]: {bad_columns}"
    )

# Protect against tiny floating-point errors.
X = X.clip(
    lower=0.0,
    upper=1.0,
)


# ============================================================
# 11. CALCULATE CRITIC STANDARD DEVIATIONS
# ============================================================

# Standard deviation represents the contrast or variability
# supplied by each criterion.
#
# ddof=0 treats the 2,619 papers as the complete decision set.
standard_deviation = X.std(
    axis=0,
    ddof=0,
)


# ============================================================
# 12. CALCULATE THE CORRELATION MATRIX
# ============================================================

correlation_matrix = X.corr(
    method="pearson"
)

# A constant criterion can produce undefined correlations.
# Such undefined off-diagonal correlations are set to zero,
# meaning the constant criterion is treated as uncorrelated.
correlation_matrix = correlation_matrix.fillna(
    0.0
)

# Each criterion is perfectly correlated with itself.
np.fill_diagonal(
    correlation_matrix.values,
    1.0,
)

correlation_matrix.to_csv(
    CORRELATION_FILE,
    index=True,
)


# ============================================================
# 13. CALCULATE CRITIC CONFLICT
# ============================================================

# For criterion j:
#
# conflict_j = sum_k (1 - r_jk)
#
# A criterion receives more conflict information when it is
# less redundant with the other criteria.
conflict = (
    1.0 - correlation_matrix
).sum(axis=1)


# ============================================================
# 14. CALCULATE INFORMATION CONTENT
# ============================================================

# CRITIC information content:
#
# C_j = sigma_j × sum_k(1 - r_jk)
#
# where:
#   sigma_j = standard deviation of criterion j
#   r_jk    = correlation between criteria j and k
information_content = (
    standard_deviation
    * conflict
)


# ============================================================
# 15. CALCULATE CRITIC WEIGHTS
# ============================================================

information_total = float(
    information_content.sum()
)

if not np.isfinite(
    information_total
):
    raise ValueError(
        "The total CRITIC information content is invalid."
    )

if information_total <= 0:
    raise ValueError(
        "CRITIC cannot calculate weights because the total "
        "information content is zero. Check whether one or "
        "more criteria are constant."
    )

critic_weights = (
    information_content
    / information_total
)

# Normalize once more to ensure numerical precision.
critic_weights = (
    critic_weights
    / critic_weights.sum()
)


# ============================================================
# 16. VERIFY THE WEIGHTS
# ============================================================

weight_sum = float(
    critic_weights.sum()
)

if not np.isclose(
    weight_sum,
    1.0,
    atol=1e-12,
):
    raise ValueError(
        "CRITIC weights do not sum to 1. "
        f"Current sum: {weight_sum}"
    )

if (
    critic_weights < -1e-12
).any():
    raise ValueError(
        "CRITIC generated at least one negative weight."
    )

critic_weights = critic_weights.clip(
    lower=0.0
)

critic_weights = (
    critic_weights
    / critic_weights.sum()
)

weight_sum = float(
    critic_weights.sum()
)


# ============================================================
# 17. CALCULATE WEIGHTED CONTRIBUTIONS
# ============================================================

df["Relevance Contribution"] = (
    X["Relevance Norm"]
    * critic_weights["Relevance Norm"]
)

df["Technical Depth Contribution"] = (
    X["Technical Depth Norm CRITIC"]
    * critic_weights["Technical Depth Norm CRITIC"]
)

df["Citation Contribution"] = (
    X["Citation Norm"]
    * critic_weights["Citation Norm"]
)

df["Query Coverage Contribution"] = (
    X["Query Coverage Norm"]
    * critic_weights["Query Coverage Norm"]
)


# ============================================================
# 18. CALCULATE FINAL SCORE
# ============================================================

df["Final Score"] = (
    df["Relevance Contribution"]
    + df["Technical Depth Contribution"]
    + df["Citation Contribution"]
    + df["Query Coverage Contribution"]
)

# Since every criterion is within [0,1] and all weights are
# non-negative and sum to 1, the Final Score must be in [0,1].
#
# Clipping only protects against tiny floating-point errors.
df["Final Score"] = df[
    "Final Score"
].clip(
    lower=0.0,
    upper=1.0,
)


# ============================================================
# 19. VERIFY FINAL SCORE
# ============================================================

final_score_min = float(
    df["Final Score"].min()
)

final_score_max = float(
    df["Final Score"].max()
)

if final_score_min < -1e-12:
    raise ValueError(
        "Final Score contains a value below zero."
    )

if final_score_max > 1 + 1e-12:
    raise ValueError(
        "Final Score contains a value above one."
    )


# ============================================================
# 20. GLOBAL RANK
# ============================================================

if "Global Rank" in df.columns:
    df = df.drop(
        columns=["Global Rank"]
    )

df["Global Rank"] = (
    df["Final Score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

df = df.sort_values(
    by=[
        "Final Score",
        ID_COL,
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(
    drop=True
)


# ============================================================
# 21. OPTIONAL FIXED SCORE CLASSES
# ============================================================

# These are fixed score intervals.
# They are not equal-frequency quartiles.

df["Score Class"] = pd.cut(
    df["Final Score"],
    bins=[
        -np.inf,
        0.25,
        0.50,
        0.75,
        np.inf,
    ],
    labels=[
        "Q1",
        "Q2",
        "Q3",
        "Q4",
    ],
    right=False,
)

df["Selected Class"] = np.where(
    df["Final Score"] >= 0.50,
    "Q3-Q4",
    "Q1-Q2",
)


# ============================================================
# 22. CREATE THE WEIGHT TABLE
# ============================================================

display_names = {
    "Relevance Norm": "Taxonomy Relevance",
    "Technical Depth Norm CRITIC": "Technical Depth",
    "Citation Norm": "Citation Impact",
    "Query Coverage Norm": "Query Coverage",
}

weight_table = pd.DataFrame({
    "Criterion Column": criteria_columns,

    "Criterion": [
        display_names[column]
        for column in criteria_columns
    ],

    "Standard Deviation": [
        float(
            standard_deviation[column]
        )
        for column in criteria_columns
    ],

    "Conflict": [
        float(
            conflict[column]
        )
        for column in criteria_columns
    ],

    "Information Content": [
        float(
            information_content[column]
        )
        for column in criteria_columns
    ],

    "CRITIC Weight": [
        float(
            critic_weights[column]
        )
        for column in criteria_columns
    ],
})

weight_table["Weight Sum"] = weight_sum

weight_table.to_csv(
    WEIGHTS_FILE,
    index=False,
)


# ============================================================
# 23. SAVE THE SCORED DATA
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
)


# ============================================================
# 24. CREATE REPORT
# ============================================================

class_counts = (
    df["Score Class"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .to_dict()
)

selected_class_counts = (
    df["Selected Class"]
    .value_counts(
        dropna=False
    )
    .to_dict()
)

report = {
    "number_of_papers": int(
        len(df)
    ),

    "input_columns": {
        "relevance": RELEVANCE_COL,
        "technical_depth": TECHNICAL_DEPTH_COL,
        "citations": CITATION_COL,
        "query_coverage": QUERY_COUNT_COL,
    },

    "normalized_criteria": {
        "relevance": "Relevance Norm",
        "technical_depth": "Technical Depth Norm CRITIC",
        "citations": "Citation Norm",
        "query_coverage": "Query Coverage Norm",
    },

    "normalization": {
        "relevance": "Min-max normalization",
        "technical_depth": "Min-max normalization",
        "citations": "log1p transformation followed by min-max normalization",
        "query_coverage": "Min-max normalization",
    },

    "weights": {
        display_names[criterion]: float(
            critic_weights[criterion]
        )
        for criterion in criteria_columns
    },

    "weight_sum": weight_sum,

    "final_score": {
        "minimum": float(
            df["Final Score"].min()
        ),
        "maximum": float(
            df["Final Score"].max()
        ),
        "mean": float(
            df["Final Score"].mean()
        ),
        "median": float(
            df["Final Score"].median()
        ),
        "standard_deviation": float(
            df["Final Score"].std(
                ddof=0
            )
        ),
    },

    "score_class_counts": {
        str(key): int(value)
        for key, value in class_counts.items()
    },

    "selected_class_counts": {
        str(key): int(value)
        for key, value in selected_class_counts.items()
    },

    "selected_class_rule": (
        "Q3-Q4 when Final Score is greater than or equal "
        "to 0.50; otherwise Q1-Q2."
    ),

    "outputs": {
        "scored_papers": str(
            OUTPUT_FILE
        ),
        "weights": str(
            WEIGHTS_FILE
        ),
        "correlation_matrix": str(
            CORRELATION_FILE
        ),
        "report": str(
            REPORT_FILE
        ),
    },
}

REPORT_FILE.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 25. PRINT RESULTS
# ============================================================

print("\nCRITIC WEIGHTS")
print("=" * 70)

for criterion in criteria_columns:
    label = display_names[criterion]
    weight = critic_weights[criterion]

    print(
        f"{label:30s}: {weight:.12f}"
    )

print("-" * 70)

print(
    f"{'Total':30s}: {critic_weights.sum():.12f}"
)


print("\nFINAL SCORE SUMMARY")
print("=" * 70)

print(
    f"Minimum: {df['Final Score'].min():.12f}"
)

print(
    f"Maximum: {df['Final Score'].max():.12f}"
)

print(
    f"Mean:    {df['Final Score'].mean():.12f}"
)

print(
    f"Median:  {df['Final Score'].median():.12f}"
)


print("\nSCORE CLASS COUNTS")
print("=" * 70)

print(
    df["Score Class"]
    .value_counts()
    .sort_index()
)


print("\nCOMBINED CLASS COUNTS")
print("=" * 70)

print(
    df["Selected Class"]
    .value_counts()
)


print("\nTOP 10 PAPERS")
print("=" * 70)

top_columns = [
    ID_COL,
    RELEVANCE_COL,
    TECHNICAL_DEPTH_COL,
    CITATION_COL,
    QUERY_COUNT_COL,
    "Relevance Norm",
    "Technical Depth Norm CRITIC",
    "Citation Norm",
    "Query Coverage Norm",
    "Final Score",
    "Global Rank",
]

print(
    df[top_columns]
    .head(10)
    .to_string(
        index=False
    )
)


print("\nSAVED FILES")
print("=" * 70)

print(
    f"Scored papers:      {OUTPUT_FILE}"
)

print(
    f"CRITIC weights:     {WEIGHTS_FILE}"
)

print(
    f"Correlation matrix: {CORRELATION_FILE}"
)

print(
    f"JSON report:        {REPORT_FILE}"
)


CRITIC WEIGHTS
Taxonomy Relevance            : 0.340120775486
Technical Depth               : 0.256081244217
Citation Impact               : 0.270433748220
Query Coverage                : 0.133364232077
----------------------------------------------------------------------
Total                         : 1.000000000000

FINAL SCORE SUMMARY
Minimum: 0.009572201672
Maximum: 0.785762391512
Mean:    0.257395216559
Median:  0.232984530203

SCORE CLASS COUNTS
Score Class
Q1    1480
Q2    1012
Q3     123
Q4       4
Name: count, dtype: int64

COMBINED CLASS COUNTS
Selected Class
Q1-Q2    2492
Q3-Q4     127
Name: count, dtype: int64

TOP 10 PAPERS
Paper ID  max_taxonomy_score  Technical Depth Norm  Max Citations  Number of Queries Matched  Relevance Norm  Technical Depth Norm CRITIC  Citation Norm  Query Coverage Norm  Final Score  Global Rank
  P00003            0.014458              0.746432           3083                          8        0.734940                     0.746432       0.890862